In [137]:
import numpy as np 
import pandas as pd 
import os

In [138]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train.head()
train.info()
train.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
count,1460.000000,1460.000000,1201.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1452.000000,1460.000000,...,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,730.500000,56.897260,70.049958,10516.828082,6.099315,5.575342,1971.267808,1984.865753,103.685262,443.639726,...,94.244521,46.660274,21.954110,3.409589,15.060959,2.758904,43.489041,6.321918,2007.815753,180921.195890
std,421.610009,42.300571,24.284752,9981.264932,1.382997,1.112799,30.202904,20.645407,181.066207,456.098091,...,125.338794,66.256028,61.119149,29.317331,55.757415,40.177307,496.123024,2.703626,1.328095,79442.502883
min,1.000000,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,34900.000000
25%,365.750000,20.000000,59.000000,7553.500000,5.000000,5.000000,1954.000000,1967.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,2007.000000,129975.000000
50%,730.500000,50.000000,69.000000,9478.500000,6.000000,5.000000,1973.000000,1994.000000,0.000000,383.500000,...,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,163000.000000
75%,1095.250000,70.000000,80.000000,11601.500000,7.000000,6.000000,2000.000000,2004.000000,166.000000,712.250000,...,168.000000,68.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,214000.000000
max,1460.000000,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,5644.000000,...,857.000000,547.000000,552.000000,508.000000,480.000000,738.000000,15500.000000,12.000000,2010.000000,755000.000000


In [139]:
train.isnull().sum().sort_values(ascending=False).head(20)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
GarageYrBlt       81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtQual          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
Condition2         0
dtype: int64

# cleaning

## cleaning_v1_semantic_imputation

In [140]:
df = train.copy()

In [141]:
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"
]

for col in none_cols:
    df[col] = df[col].fillna("None")

In [142]:
df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median())
) 
df["MasVnrType"] = df["MasVnrType"].fillna("None")
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

In [143]:
print("Remaining missing values:", df.isnull().sum().sum())

Remaining missing values: 0


In [144]:
!pip install dagshub mlflow
import dagshub
dagshub.init(repo_owner='smama23', repo_name='MLassignment1', mlflow=True)

import mlflow

with mlflow.start_run(run_name="cleaning_v1"):
    mlflow.log_param("missing_strategy", "semantic + group_median")
    mlflow.log_param("LotFrontage", "median_by_neighborhood")
    mlflow.log_param("MasVnrArea", "fill_0")
    mlflow.log_param("GarageYrBlt", "fill_0")
    mlflow.log_param("Electrical", "mode")

    mlflow.log_metric("remaining_missing", df.isnull().sum().sum())

Initialized MLflow to track repo "smama23/MLassignment1"

Repository smama23/MLassignment1 initialized!

🏃 View run cleaning_v1 at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0/runs/8b17adb9b1cd4ce099ee529234a4ff38
🧪 View experiment at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0


In [145]:
df.isnull().sum().sort_values(ascending=False).head(10)

Id             0
MSSubClass     0
MSZoning       0
LotFrontage    0
LotArea        0
Street         0
Alley          0
LotShape       0
LandContour    0
Utilities      0
dtype: int64

# Feature Engineering

In [146]:
df_encoded = df.copy()

In [147]:
qual_map = {
    "None": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
}

In [148]:
qual_cols = [
    "ExterQual", "ExterCond",
    "BsmtQual", "BsmtCond",
    "HeatingQC",
    "KitchenQual",
    "FireplaceQu",
    "GarageQual", "GarageCond",
    "PoolQC"
]

for col in qual_cols:
    df_encoded[col] = df_encoded[col].map(qual_map)

In [149]:
df_encoded[qual_cols].head()

,ExterQual,ExterCond,BsmtQual,BsmtCond,HeatingQC,KitchenQual,FireplaceQu,GarageQual,GarageCond,PoolQC
0,4,3,4,3,5,4,0,3,3,0
1,3,3,4,3,5,3,3,3,3,0
2,4,3,4,3,5,4,3,3,3,0
3,3,3,3,4,4,4,4,3,3,0
4,4,3,4,3,5,4,3,3,3,0


In [150]:
with mlflow.start_run(run_name="encoding_v1"):
    mlflow.log_param("encoding", "ordinal_quality_mapping_0_to_5")

🏃 View run encoding_v1 at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0/runs/9b308e90be874534a33bbe50bc234554
🧪 View experiment at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0


In [151]:
df_encoded.select_dtypes(include="object").columns

Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'Foundation', 'BsmtExposure',
       'BsmtFinType1', 'BsmtFinType2', 'Heating', 'CentralAir', 'Electrical',
       'Functional', 'GarageType', 'GarageFinish', 'PavedDrive', 'Fence',
       'MiscFeature', 'SaleType', 'SaleCondition'],
      dtype='object')

In [152]:
df_final = df_encoded.copy()

In [153]:
df_final = pd.get_dummies(df_final)

In [154]:
df_final = df_final[
    ~((df_final["GrLivArea"] > 4000) & (df_final["SalePrice"] < 200000))
].copy()

print("Removed outliers. New shape:", df_final.shape)

Removed outliers. New shape: (1458, 264)


In [155]:
print(df_final.shape)

(1458, 264)


In [156]:
df_final.select_dtypes(include="object").columns

Index([], dtype='object')

In [157]:
with mlflow.start_run(run_name="encoding_v2_onehot"):
    mlflow.log_param("encoding", "one_hot_all_nominal")
    mlflow.log_metric("num_features", df_final.shape[1])

🏃 View run encoding_v2_onehot at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0/runs/900e2d2aa80945d3ac4beb8bf8207b8c
🧪 View experiment at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0


In [158]:
X = df_final.drop("SalePrice", axis=1)
y = df_final["SalePrice"]

In [159]:
import numpy as np

y = np.log1p(y)

In [160]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [161]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [162]:
y_pred = model.predict(X_val)

In [163]:
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print("RMSE:", rmse)

RMSE: 0.1301446343267266


In [164]:
import mlflow

with mlflow.start_run(run_name="baseline_linear_regression"):
    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("features", df_final.shape[1])
    mlflow.log_metric("rmse", rmse)

🏃 View run baseline_linear_regression at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0/runs/82ba30321bd645a28159444ddd2b8331
🧪 View experiment at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0


In [165]:
rmse

np.float64(0.1301446343267266)

In [166]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=10)
ridge.fit(X_train, y_train)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",10
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",None


In [167]:
y_pred_ridge = ridge.predict(X_val)

In [168]:
rmse_ridge = np.sqrt(mean_squared_error(y_val, y_pred_ridge))
print("Ridge RMSE:", rmse_ridge)

Ridge RMSE: 0.12194757957385126


In [169]:
with mlflow.start_run(run_name="ridge_alpha_10"):
    mlflow.log_param("model", "Ridge")
    mlflow.log_param("alpha", 10)
    mlflow.log_metric("rmse", rmse_ridge)

🏃 View run ridge_alpha_10 at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0/runs/e56ebea6e02c44ed8306a5f8ae97e5d8
🧪 View experiment at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0


# Feature Selection

In [170]:
import pandas as pd
import numpy as np

corr = pd.concat([X_train, y_train], axis=1).corr()["SalePrice"].abs()

selected_features = corr[corr > 0.05].index.drop("SalePrice")

X_train_sel = X_train[selected_features]
X_val_sel = X_val[selected_features]

In [171]:
ridge_sel = Ridge(alpha=10)
ridge_sel.fit(X_train_sel, y_train)

y_pred_sel = ridge_sel.predict(X_val_sel)

rmse_sel = np.sqrt(mean_squared_error(y_val, y_pred_sel))
print("Selected Features RMSE:", rmse_sel)

Selected Features RMSE: 0.12599562603692


In [172]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import numpy as np

alphas = [0.1, 1, 5, 10, 50, 100]

results = []

for a in alphas:
    model = Ridge(alpha=a)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    results.append((a, rmse))

results

[(0.1, np.float64(0.12781351624093948)),
 (1, np.float64(0.12255920837103478)),
 (5, np.float64(0.12161710552233268)),
 (10, np.float64(0.12194757957385126)),
 (50, np.float64(0.12347096092482458)),
 (100, np.float64(0.12515123378106996))]

In [173]:
with mlflow.start_run(run_name="ridge_best_alpha_0_1"):
    mlflow.log_param("model", "Ridge")
    mlflow.log_param("alpha", 0.1)
    mlflow.log_metric("rmse", 0.12409264369272437)

🏃 View run ridge_best_alpha_0_1 at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0/runs/a01781bde37c465796a60b088c25afae
🧪 View experiment at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0


In [174]:
X_manual = X_train.copy()
X_val_manual = X_val.copy()

In [175]:
drop_cols = [
    "Alley",
    "PoolQC",
    "Fence",
    "MiscFeature",
    "Id"
]

X_manual = X_manual.drop(columns=drop_cols, errors="ignore")
X_val_manual = X_val_manual.drop(columns=drop_cols, errors="ignore")

In [176]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import numpy as np

model_manual = Ridge(alpha=0.1)
model_manual.fit(X_manual, y_train)

preds_manual = model_manual.predict(X_val_manual)

rmse_manual = np.sqrt(mean_squared_error(y_val, preds_manual))

print("Manual Drop RMSE:", rmse_manual)

Manual Drop RMSE: 0.12788150180168742


In [177]:
with mlflow.start_run(run_name="manual_feature_drop_ridge"):
    mlflow.log_param("features_removed", drop_cols)
    mlflow.log_param("model", "Ridge")
    mlflow.log_param("alpha", 0.1)
    mlflow.log_metric("rmse", rmse_manual)

🏃 View run manual_feature_drop_ridge at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0/runs/2b0cdc0f970f48b996046225ba165fdd
🧪 View experiment at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0


## Feature Engineering v3 - Interaction Features

In [178]:
df_inter = df.copy()

In [179]:
df_inter = df.copy()

df_inter["HouseAge"] = df_inter["YrSold"] - df_inter["YearBuilt"]
df_inter["RemodAge"] = df_inter["YrSold"] - df_inter["YearRemodAdd"]

df_inter["OverallQual_GrLivArea"] = df_inter["OverallQual"] * df_inter["GrLivArea"]
df_inter["BsmtQual_TotalBsmtSF"] = df_inter["BsmtQual"] * df_inter["TotalBsmtSF"]
df_inter["GarageStrength"] = df_inter["GarageCars"] * df_inter["GarageArea"]
df_inter["HouseAge_Quality"] = df_inter["HouseAge"] * df_inter["OverallQual"]
df_inter["Remod_Quality"] = df_inter["RemodAge"] * df_inter["OverallQual"]

In [180]:
df_inter["TotalSF"] = (
    df_inter["TotalBsmtSF"] +
    df_inter["1stFlrSF"] +
    df_inter["2ndFlrSF"]
)

In [181]:
df_inter = pd.get_dummies(df_inter)

In [182]:
import numpy as np

X = df_inter.drop("SalePrice", axis=1)
y = np.log1p(df_inter["SalePrice"])

In [183]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [184]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import numpy as np

model = Ridge(alpha=0.1)
model.fit(X_train, y_train)

preds = model.predict(X_val)

rmse_inter = np.sqrt(mean_squared_error(y_val, preds))

print("Interaction Features RMSE:", rmse_inter)

Interaction Features RMSE: 0.14177473668913407


In [185]:
import mlflow

with mlflow.start_run(run_name="ridge_interaction_features_v3"):
    mlflow.log_param("feature_engineering", "interaction_features_v3")
    mlflow.log_param("features_added", [
        "OverallQual_GrLivArea",
        "BsmtQual_TotalBsmtSF",
        "GarageStrength",
        "HouseAge_Quality",
        "Remod_Quality"
    ])
    mlflow.log_param("model", "Ridge")
    mlflow.log_param("alpha", 0.1)
    mlflow.log_metric("rmse", rmse_inter)

🏃 View run ridge_interaction_features_v3 at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0/runs/e9b515776da34ade9a5dbc1fda55ef4c
🧪 View experiment at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0


In [186]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

In [187]:
gbr = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gbr.fit(X_train, y_train)

,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",300
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are""friedman_mse"" for the mean squared error with improvement score byFriedman, ""squared_error"" for mean squared error. The default value of""friedman_mse"" is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft

In [188]:
preds = gbr.predict(X_val)

rmse_gbr = np.sqrt(mean_squared_error(y_val, preds))

print("Gradient Boosting RMSE:", rmse_gbr)

Gradient Boosting RMSE: 0.13624247356628635


In [189]:
import mlflow

with mlflow.start_run(run_name="gradient_boosting_v1"):
    mlflow.log_param("model", "GradientBoostingRegressor")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 3)

    mlflow.log_metric("rmse", rmse_gbr)

🏃 View run gradient_boosting_v1 at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0/runs/3498b2c826654f968e661c0423e522a0
🧪 View experiment at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0


In [190]:
%pip install xgboost
from xgboost import XGBRegressor 

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train, y_train)

Note: you may need to restart the kernel to use updated packages.


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [191]:
preds_xgb = xgb.predict(X_val)

rmse_xgb = np.sqrt(mean_squared_error(y_val, preds_xgb))

print("XGBoost RMSE:", rmse_xgb)

XGBoost RMSE: 0.13490211466466867


In [192]:
with mlflow.start_run(run_name="xgboost_v1"):
    mlflow.log_param("model", "XGBRegressor")
    mlflow.log_param("n_estimators", 500)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 4)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)

    mlflow.log_metric("rmse", rmse_xgb)

🏃 View run xgboost_v1 at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0/runs/443bc6e0e0d74fe9b39ebb2e8714fe37
🧪 View experiment at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0


In [193]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

In [194]:
rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",400
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [195]:
preds_rf = rf.predict(X_val)

rmse_rf = np.sqrt(mean_squared_error(y_val, preds_rf))

print("Random Forest RMSE:", rmse_rf)

Random Forest RMSE: 0.14559789393237513


In [196]:
import mlflow

with mlflow.start_run(run_name="random_forest_v1"):
    mlflow.log_param("model", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 400)
    mlflow.log_param("max_depth", None)
    mlflow.log_param("min_samples_split", 2)
    mlflow.log_param("min_samples_leaf", 1)

    mlflow.log_metric("rmse", rmse_rf)

🏃 View run random_forest_v1 at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0/runs/0956afeef1544538b6ff01a1f58796ca
🧪 View experiment at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0


In [197]:
from sklearn.linear_model import Ridge

X_full = df_final.drop("SalePrice", axis=1)
X_full = X_full.drop(columns=drop_cols, errors="ignore")
y_full = np.log1p(df_final["SalePrice"])

final_model = Ridge(alpha=10.0)
final_model.fit(X_full, y_full)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",10.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",None


In [198]:
print("Final model trained on full data.")
print("Estimated RMSE from K-Fold: 0.12505")

Final model trained on full data.
Estimated RMSE from K-Fold: 0.12505


In [199]:
import mlflow
import mlflow.sklearn

with mlflow.start_run(run_name="ridge_manual_drop_final"):
    mlflow.log_param("model", "Ridge")
    mlflow.log_param("alpha", 10)
    mlflow.log_param("feature_selection", "manual_drop")
    mlflow.log_metric("cv_mean_rmse", 0.12505)  

    mlflow.sklearn.log_model(
        sk_model=final_model,
        name="model",
        registered_model_name="house_prices_ridge_manual"
    )

2026/04/12 13:33:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'house_prices_ridge_manual' already exists. Creating a new version of this model...
2026/04/12 13:33:19 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: house_prices_ridge_manual, version 20
Created version '20' of model 'house_prices_ridge_manual'.


🏃 View run ridge_manual_drop_final at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0/runs/ff054f5333e74043a7ec6dd5eaaf2d01
🧪 View experiment at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0


In [200]:
import mlflow.sklearn

with mlflow.start_run(run_name="ridge_final_model"):
    mlflow.log_param("model", "Ridge")
    mlflow.log_param("alpha", 10)
    mlflow.log_metric("rmse", rmse)
    
    mlflow.sklearn.log_model(
        final_model,
        name="ridge_model",
        registered_model_name="RidgeHousePrices" 
    )

2026/04/12 13:33:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'RidgeHousePrices' already exists. Creating a new version of this model...
2026/04/12 13:33:43 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RidgeHousePrices, version 9
Created version '9' of model 'RidgeHousePrices'.


🏃 View run ridge_final_model at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0/runs/58a87991eddc4c309a05998bcaeee2f4
🧪 View experiment at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0


In [201]:
import mlflow.sklearn
import joblib
import os

with mlflow.start_run(run_name="ridge_final_model"):
    mlflow.log_param("model", "Ridge")
    mlflow.log_param("alpha", 10)
    mlflow.log_metric("rmse", rmse)
    
    mlflow.sklearn.log_model(
        final_model,
        name="ridge_model",
        registered_model_name="RidgeHousePrices"
    )
    
    joblib.dump(list(X_full.columns), "model_columns.pkl")
    mlflow.log_artifact("model_columns.pkl")

2026/04/12 13:33:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'RidgeHousePrices' already exists. Creating a new version of this model...
2026/04/12 13:34:06 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RidgeHousePrices, version 10
Created version '10' of model 'RidgeHousePrices'.


🏃 View run ridge_final_model at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0/runs/2c5903b123e6470b8d736bb29bd073ea
🧪 View experiment at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0


# KFold

In [202]:
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import numpy as np

X_full_kf = df_final.drop("SalePrice", axis=1)
X_full_kf = X_full_kf.drop(columns=drop_cols, errors="ignore")
y_full_kf = np.log1p(df_final["SalePrice"])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_folds = []

for alpha in [0.1, 1.0, 10.0, 50.0]:
    rmse_folds = []
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_full), 1):
        X_tr, X_val_kf = X_full.iloc[train_idx], X_full.iloc[val_idx]
        y_tr, y_val_kf = y_full.iloc[train_idx], y_full.iloc[val_idx]

        model_kf = Ridge(alpha=alpha)
        model_kf.fit(X_tr, y_tr)
        preds_kf = model_kf.predict(X_val_kf)
        rmse_folds.append(np.sqrt(mean_squared_error(y_val_kf, preds_kf)))

    print(f"Alpha {alpha:5} -> Mean RMSE: {np.mean(rmse_folds):.5f}  Std: {np.std(rmse_folds):.5f}")

Alpha   0.1 -> Mean RMSE: 0.12505  Std: 0.00924
Alpha   1.0 -> Mean RMSE: 0.11886  Std: 0.00701
Alpha  10.0 -> Mean RMSE: 0.11483  Std: 0.00708
Alpha  50.0 -> Mean RMSE: 0.11575  Std: 0.00824


In [203]:
with mlflow.start_run(run_name="kfold_cv_ridge_manual"):
    mlflow.log_param("model", "Ridge")
    mlflow.log_param("alpha", 10)
    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("feature_selection", "manual_drop")
    mlflow.log_metric("cv_mean_rmse", np.mean(rmse_folds))
    mlflow.log_metric("cv_std_rmse", np.std(rmse_folds))

🏃 View run kfold_cv_ridge_manual at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0/runs/51b634afb44342d5b0bc0d065c431f1b
🧪 View experiment at: https://dagshub.com/smama23/MLassignment1.mlflow/#/experiments/0
